# Gold v2 dev 세트 — 기준선 (agent 설계용)

dev 세트(supervised train, 약 240장 / Q1·Q2 약 480문항)에서 agent 비교 기준이 될 고정 조건 3개를 돌립니다.
논문 수치는 gold v2로만 내고, 이 결과는 agent 구성을 고르는 기준입니다.

- `image_only`: target만 (gold baseline과 같은 프롬프트)
- `normal_ref_fixed`: [자산별 고정 정상 이미지, target] (gold normal reference와 같은 규칙·프롬프트)
- `normal_ref_retrieved`: [DINOv2로 찾은 비슷한 정상 이미지, target] (같은 프롬프트)
- 모델: InternVL3.5-8B, Qwen3.5-4B (설계 단계 2개)
- 지표: Q1 F1, Q2 macro-F1, Q1·Q2 모순, Joint, FDR (전체·자산별) + 고정 vs 검색 reference 등 paired bootstrap 95% CI

GPU: **L4** (4bit 금지). 호출 약 480 × 3조건 × 2모델 = 2,880회, 1~1.5시간. 20문항마다 Drive 저장, 끊기면 다시 실행하면 이어집니다.
결과: `MyDrive/lvlm_insplad_results/gold_v2_dev_baselines/` → GitHub `results/latest/gold_v2_dev_baselines/`

In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
DEV_FOLDER = "lvlm_insplad_qa/gold_v2_dev"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_dev_baselines"
MODELS = ["internvl3_5_8b","qwen3_5_4b"]
CONDITIONS = ["image_only","normal_ref_fixed","normal_ref_retrieved"]
CHECKPOINT_EVERY = 20
PUBLISH = True

UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
UNSUP_ZIP_PATH = ""   # 비우면 자동 검색
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""     # 비우면 자동 검색

In [ ]:
# Drive 연결 + repo clone + 패키지
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
DEV_DIR = DRIVE_ROOT / DEV_FOLDER
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
RUN_ROOT.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd, *, env=None, check=True):
    """Run a command and stream its output into this cell (Colab hides plain subprocess output)."""
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""), flush=True)
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-60:]
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed with exit code {p.returncode} (마지막 출력 위 참고)")
    return p.returncode

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({"GIT_CONFIG_COUNT":"1", "GIT_CONFIG_KEY_0":"http.extraHeader",
                "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}", "GIT_TERMINAL_PROMPT":"0"})
run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env
run([sys.executable,"-m","pip","install","-q","transformers==5.14.0","accelerate==1.14.0",
     "sentencepiece==0.2.2","pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66","timm"])
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))
import pandas as pd
import torch
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
if not torch.cuda.is_available():
    raise RuntimeError("GPU Runtime(L4/A100)을 사용하세요.")

In [ ]:
def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음)", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
UNSUP_ZIP = find_zip(UNSUP_ZIP_NAME, UNSUP_ZIP_SIZE, UNSUP_ZIP_PATH)
print("SUP:", SUP_ZIP); print("UNSUP:", UNSUP_ZIP)
summary = json.loads((DEV_DIR / "build_summary.json").read_text(encoding="utf-8"))
if summary.get("quality_status") != "FROZEN":
    raise RuntimeError(f"dev 세트가 FROZEN이 아닙니다: {summary.get('quality_status')} — dev 세트 노트북을 먼저 끝내세요")
DEV_QA = DEV_DIR / summary["file"]
print("dev:", summary["images"], "장 /", summary["questions"], "문항 | 예상 호출", summary["questions"] * len(CONDITIONS) * len(MODELS), "회")

In [ ]:
# 실행 — 끊기면 이 셀만 다시 실행해도 완료된 문항은 재사용
run([sys.executable, REPO_DIR / "scripts/run_gold_v2_dev_baselines.py",
     "--dev-qa", DEV_QA, "--sup-zip", SUP_ZIP, "--unsup-zip", UNSUP_ZIP, "--out-dir", RUN_ROOT,
     "--models", *MODELS, "--conditions", *CONDITIONS, "--checkpoint-every", str(CHECKPOINT_EVERY)])

In [ ]:
# 결과
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 200)
lb = pd.read_csv(RUN_ROOT / "leaderboard.csv")
display(lb[["model_key","condition","q1_anomaly_f1","q2_defect_macro_f1","consistency_error","joint_accuracy","false_defect_rate"]].round(3))
asset = pd.read_csv(RUN_ROOT / "metrics_by_asset.csv")
print("\n[자산별 Joint Acc]")
display(asset.pivot_table(index=["model_key","condition"], columns="asset_category", values="joint_accuracy").round(3))
sig = pd.read_csv(RUN_ROOT / "significance.csv")
print("\n[paired bootstrap] delta (CI가 0을 포함하지 않으면 significant)")
display(sig.pivot_table(index=["model_key","baseline","condition"], columns="metric", values="delta").round(3))
print("significant:", int(sig.significant.sum()), "/", len(sig))
status = json.loads((RUN_ROOT / "run_status.json").read_text(encoding="utf-8"))
print("row_errors 합계:", sum(s["row_errors"] for s in status), "(0이 아니면 실행 셀을 다시 실행)")

In [ ]:
if PUBLISH:
    run([sys.executable, REPO_DIR / "scripts/publish_results_to_github.py", "--out-dir", RUN_ROOT,
         "--repo", REPO, "--branch", REPO_REF, "--prefix", "gold_v2_dev_baselines",
         "--files", "leaderboard.csv", "metrics_by_asset.csv", "significance.csv", "run_status.json",
         "run_metadata.json"], env=os.environ.copy())
    print("GitHub: results/latest/gold_v2_dev_baselines/")